**ETL**

**0. Importando bibliotecas**

In [1]:
import os
from pathlib import Path
import unicodedata
import pandas as pd

**1. Extraindo e juntando os dados**

Aqui os arquivos são carregados e os dados são colocados em DataFrames do pandas.

Antes de rodar é necessario extrair os arquivos "*.rar" da pasta "Dados" primeiro.

In [2]:
# Caminho dos arquivos com os dados
arquivos_path = Path("..") / "Datas"
arquivos_name_list = sorted([arquivo.name for arquivo in arquivos_path.glob("*.csv")])

# Carrega os arquivos como DataFrames do pandas
# Mantemos tudo como texto na leitura para preservar CNPJs e padronizar a limpeza depois.
tdf_dict = {}
for arquivo_name in arquivos_name_list:
    try:
        path = arquivos_path / arquivo_name
        tdf = pd.read_csv(path, sep=";", encoding="cp1252", dtype=str)
        tdf["arquivo_origem"] = arquivo_name
        tdf_dict[arquivo_name] = tdf
    except Exception as e:
        print(f"Erro ao ler o arquivo '{arquivo_name}': {e}")

# Junta os dados em um unico DataFrame:
df = pd.concat(tdf_dict.values(), ignore_index=True)

Foram utilizados os dados de 2024 a 2026. O arquivo de 2026 ainda pode estar em atualização, então vale conferir o volume antes de usar comparações finais.

**2. Analize inicial dos dados**

**2.1. Formato original dos dados:**

<pre style="font-family: monospace; white-space: pre;">
RangeIndex: 3029329 entries, 0 to 3029328
Data columns (total 18 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   DatGeracaoConjuntoDados     str    
 1   NumCNPJAgenteDistribuidora  int64  
 2   SigAgenteDistribuidora      str    
 3   NomAgenteDistribuidora      str    
 4   NomTipoMercado              str    
 5   DscModalidadeTarifaria      str    
 6   DscSubGrupoTarifario        str    
 7   DscClasseConsumoMercado     str    
 8   DscSubClasseConsumidor      str    
 9   DscDetalheConsumidor        str    
 10  IdeAgenteAcessante          float64
 11  NumCNPJAgenteAcessante      float64
 12  NomAgenteAcessante          str    
 13  DscPostoTarifario           str    
 14  DscOpcaoEnergia             str    
 15  DscDetalheMercado           str    
 16  DatCompetencia              str    
 17  VlrMercado                  str       
dtypes: float64(2), int64(1), str(15)
</pre>

Inicialmente é possivel observar que o conjunto de dados tem 3.029.329 linhas e 18 colunas;

Originalmente havia 2 colunas sendo do tipo float64 (IdeAgenteAcessante e NumCNPJAgenteAcessante), 1 do tipo int64 (NumCNPJAgenteDistribuidora) e as outras 15 colunas sendo do tipo string.

**2.2. Dados após extração e junção**

Todas as colunas foram trasnformadas em strings para padronizar a limpeza e para preservar os CNPJs, que após a leitura estavam sendo convertidos em numeros no formato de de notação científica.

A coluna "arquivo_origem" foi adicionada durante a criação do DataFrame para identificar de qual arquivo cada registro veio.

In [3]:
df.head(6)

,DatGeracaoConjuntoDados,NumCNPJAgenteDistribuidora,SigAgenteDistribuidora,NomAgenteDistribuidora,NomTipoMercado,DscModalidadeTarifaria,DscSubGrupoTarifario,DscClasseConsumoMercado,DscSubClasseConsumidor,DscDetalheConsumidor,IdeAgenteAcessante,NumCNPJAgenteAcessante,NomAgenteAcessante,DscPostoTarifario,DscOpcaoEnergia,DscDetalheMercado,DatCompetencia,VlrMercado,arquivo_origem
0,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Sistema Isolado - Regular,Convencional,B1,Residencial,Residencial baixa renda – faixa 04,Não se aplica,NaN,37121669900,Não se aplica,Não se aplica,CATIVO,Energia TE (kWh),2024-01-01,"583151,000000",samp-2024.csv
1,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Regular,Convencional,B1,Residencial,Residencial baixa renda – faixa 01,Não se aplica,NaN,37121669900,Não se aplica,Não se aplica,CATIVO,PIS/PASEP (R$),2024-01-01,"5431,290000",samp-2024.csv
2,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Regular,Azul,A4,Serviço público,"Água, esgoto e saneamento",Não se aplica,NaN,37121669900,Não se aplica,Ponta,CATIVO,Receita Demanda (R$),2024-01-01,"28137,260000",samp-2024.csv
3,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Sistema de Compensação GD II,Branca,B1,Residencial,Residencial,Não se aplica,NaN,37121669900,Não se aplica,Fora ponta,CATIVO,Receita energia compensada (R$),2024-01-01,"235,940000",samp-2024.csv
4,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Regular,Convencional,B1,Residencial,Residencial baixa renda – faixa 04,Não se aplica,NaN,37121669900,Não se aplica,Não se aplica,CATIVO,Receita Energia (R$),2024-01-01,"2061470,440000",samp-2024.csv
5,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Refaturamento - Regular,Convencional,B3,Comercial,Não se aplica,Não se aplica,NaN,37121669900,Não se aplica,Não se aplica,CATIVO,ICMS (R$),2024-01-01,"-418,830000",samp-2024.csv


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3029329 entries, 0 to 3029328
Data columns (total 19 columns):
 #   Column                      Dtype
---  ------                      -----
 0   DatGeracaoConjuntoDados     str  
 1   NumCNPJAgenteDistribuidora  str  
 2   SigAgenteDistribuidora      str  
 3   NomAgenteDistribuidora      str  
 4   NomTipoMercado              str  
 5   DscModalidadeTarifaria      str  
 6   DscSubGrupoTarifario        str  
 7   DscClasseConsumoMercado     str  
 8   DscSubClasseConsumidor      str  
 9   DscDetalheConsumidor        str  
 10  IdeAgenteAcessante          str  
 11  NumCNPJAgenteAcessante      str  
 12  NomAgenteAcessante          str  
 13  DscPostoTarifario           str  
 14  DscOpcaoEnergia             str  
 15  DscDetalheMercado           str  
 16  DatCompetencia              str  
 17  VlrMercado                  str  
 18  arquivo_origem              str  
dtypes: str(19)
memory usage: 439.1 MB


Segue uma descrição sobre cada coluna e o que ela representa:

00. DatGeracaoConjuntoDados:
01. NumCNPJAgenteDistribuidora: CNPJ da agencia distribuidora de energia.
02. SigAgenteDistribuidora: Sigla utilizada para idantificar a agencia distribuidora de energia.
03. NomAgenteDistribuidora: Nome completo da agencia distribuidora de energia.
04. NomTipoMercado: Identifica qual o tipo de mercado de energia elétrica do registro.
05. DscModalidadeTarifaria: Indica o tipo de cobrança aplicada ao fornecimento de energia elétrica daquele registro.
06. DscSubGrupoTarifario: Indica o subgrupo tarifário ao qual a modalidade tarifaria faz parte.
07. DscClasseConsumoMercado: 
08. DscSubClasseConsumidor:
09. DscDetalheConsumidor:
10. IdeAgenteAcessante:
11. NumCNPJAgenteAcessante:
12. NomAgenteAcessante:
13. DscPostoTarifario: indica o período do dia ao qual o consumo ou a demanda de energia está associado.
14. DscOpcaoEnergia:
15. DscDetalheMercado: Indica o que o registro está medindo no VlrMercado.
16. DatCompetencia:
17. VlrMercado:
18. arquivo_origem: Arquivo de onde o registro (linha do DataFrame) veio. 

**3. Tratamento e correções iniciais dos dados**

Aqui é feito um tratamento inicial para eliminar espaços extras e deixar os valores das colunas com a tipagem correta e em um formato padronizado.

**3.1. Removendo espaços extras no inicio e no fim**

In [5]:
# Padroniza nomes de colunas e remove espacos extras no inicio e no final do nome de todas as colunas:
df.columns = df.columns.str.strip()

In [6]:
# Remove os espaços extras no inicio e no final dos valores nas colunas:
text_columns = [
    coluna for coluna in df.columns
    if pd.api.types.is_string_dtype(df[coluna]) or df[coluna].dtype == object
]
for coluna in text_columns:
    df[coluna] = df[coluna].str.strip()

**3.2. Trocando tipo dos dados**

In [7]:
# Converete campos de data do tipo string para o tipo datetime.
df["DatGeracaoConjuntoDados"] = pd.to_datetime(df["DatGeracaoConjuntoDados"], errors="coerce")
df["DatCompetencia"] = pd.to_datetime(df["DatCompetencia"], errors="coerce")

In [8]:
# Trasforma valores da coluna "VlrMercado" de strings para numeros.
df["VlrMercado"] = pd.to_numeric(
    df["VlrMercado"].str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
    errors="coerce",
)

**3.3. Padronizando formato dos dados**

In [9]:
# Nas colunas de CNPJ, remove dos CNPJs qualquer caracter que não for um digito.
# CNPJ continua sendo string.
for coluna in ["NumCNPJAgenteDistribuidora", "NumCNPJAgenteAcessante"]:
    df[coluna] = (
        df[coluna]
        .str.replace(r"\D", "", regex=True)
        .replace({"": pd.NA})
    )

**3.4. DataFrame após o tratamento**

In [10]:
df.head(6)

,DatGeracaoConjuntoDados,NumCNPJAgenteDistribuidora,SigAgenteDistribuidora,NomAgenteDistribuidora,NomTipoMercado,DscModalidadeTarifaria,DscSubGrupoTarifario,DscClasseConsumoMercado,DscSubClasseConsumidor,DscDetalheConsumidor,IdeAgenteAcessante,NumCNPJAgenteAcessante,NomAgenteAcessante,DscPostoTarifario,DscOpcaoEnergia,DscDetalheMercado,DatCompetencia,VlrMercado,arquivo_origem
0,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Sistema Isolado - Regular,Convencional,B1,Residencial,Residencial baixa renda – faixa 04,Não se aplica,NaN,37121669900,Não se aplica,Não se aplica,CATIVO,Energia TE (kWh),2024-01-01,583151.00,samp-2024.csv
1,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Regular,Convencional,B1,Residencial,Residencial baixa renda – faixa 01,Não se aplica,NaN,37121669900,Não se aplica,Não se aplica,CATIVO,PIS/PASEP (R$),2024-01-01,5431.29,samp-2024.csv
2,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Regular,Azul,A4,Serviço público,"Água, esgoto e saneamento",Não se aplica,NaN,37121669900,Não se aplica,Ponta,CATIVO,Receita Demanda (R$),2024-01-01,28137.26,samp-2024.csv
3,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Sistema de Compensação GD II,Branca,B1,Residencial,Residencial,Não se aplica,NaN,37121669900,Não se aplica,Fora ponta,CATIVO,Receita energia compensada (R$),2024-01-01,235.94,samp-2024.csv
4,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Regular,Convencional,B1,Residencial,Residencial baixa renda – faixa 04,Não se aplica,NaN,37121669900,Não se aplica,Não se aplica,CATIVO,Receita Energia (R$),2024-01-01,2061470.44,samp-2024.csv
5,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,Refaturamento - Regular,Convencional,B3,Comercial,Não se aplica,Não se aplica,NaN,37121669900,Não se aplica,Não se aplica,CATIVO,ICMS (R$),2024-01-01,-418.83,samp-2024.csv


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3029329 entries, 0 to 3029328
Data columns (total 19 columns):
 #   Column                      Dtype         
---  ------                      -----         
 0   DatGeracaoConjuntoDados     datetime64[us]
 1   NumCNPJAgenteDistribuidora  str           
 2   SigAgenteDistribuidora      str           
 3   NomAgenteDistribuidora      str           
 4   NomTipoMercado              str           
 5   DscModalidadeTarifaria      str           
 6   DscSubGrupoTarifario        str           
 7   DscClasseConsumoMercado     str           
 8   DscSubClasseConsumidor      str           
 9   DscDetalheConsumidor        str           
 10  IdeAgenteAcessante          str           
 11  NumCNPJAgenteAcessante      str           
 12  NomAgenteAcessante          str           
 13  DscPostoTarifario           str           
 14  DscOpcaoEnergia             str           
 15  DscDetalheMercado           str           
 16  DatCompetencia              d

**4. Camada Silver - limpeza e padronização**

Nesta etapa tratamos valores nulos, removemos duplicidades, padronizamos textos e criamos colunas auxiliares para análise.

**4.1. Verificando valores vazios ou nulos**

In [12]:
# Verifica ocorrencia de strings vazias por coluna:
mask = df.astype(str).apply(lambda col: col.str.strip() == r'^\s*$')
print(mask.sum())

DatGeracaoConjuntoDados       0
NumCNPJAgenteDistribuidora    0
SigAgenteDistribuidora        0
NomAgenteDistribuidora        0
NomTipoMercado                0
DscModalidadeTarifaria        0
DscSubGrupoTarifario          0
DscClasseConsumoMercado       0
DscSubClasseConsumidor        0
DscDetalheConsumidor          0
IdeAgenteAcessante            0
NumCNPJAgenteAcessante        0
NomAgenteAcessante            0
DscPostoTarifario             0
DscOpcaoEnergia               0
DscDetalheMercado             0
DatCompetencia                0
VlrMercado                    0
arquivo_origem                0
dtype: int64


Não há nenhuma string vazia no conjunto de dados, logo não é necessario tratar  isso.

In [13]:
#Verificando valores nulos por coluna:
print("Valores ausentes por coluna:")
print(df.isnull().sum())

Valores ausentes por coluna:
DatGeracaoConjuntoDados             0
NumCNPJAgenteDistribuidora          0
SigAgenteDistribuidora              0
NomAgenteDistribuidora              0
NomTipoMercado                      0
DscModalidadeTarifaria              0
DscSubGrupoTarifario                0
DscClasseConsumoMercado             0
DscSubClasseConsumidor              0
DscDetalheConsumidor                0
IdeAgenteAcessante            3029329
NumCNPJAgenteAcessante         173586
NomAgenteAcessante                  0
DscPostoTarifario                   0
DscOpcaoEnergia                     0
DscDetalheMercado                   0
DatCompetencia                      0
VlrMercado                          0
arquivo_origem                      0
dtype: int64


Apenas 2 colunas tem valores nulos que precisam ser tratados, "IdeAgenteAcessante" e "NumCNPJAgenteAcessante".

**4.2. Verificando e removendo linhas duplicadas**

Encontra e remove linhas repetidas, que tem exatamente os mesmos valores em todas as colunas.

In [14]:
# Copia banco de dados:
silver_df = df.copy()

# Mostra quantas linhas duplicadas existem antes de remover:
linhas_duplicadas_exatas = int(silver_df.duplicated().sum())
print("Numero de linhas duplicadas (antes):", linhas_duplicadas_exatas)

# Encontra linhas duplicadas e remove elas:
silver_df_sem_dup = silver_df.drop_duplicates().reset_index(drop=True)

# Mostra quantas linhas duplicadas existem depois de remover:
linhas_duplicadas_exatas_depois = int(silver_df_sem_dup.duplicated().sum())
print("Numero de linhas duplicadas (depois):", linhas_duplicadas_exatas_depois)

# Registra o DataFrame que será usado daqui para frente como o dataframe sem linhas com duplicidades exatas:
silver_df = silver_df_sem_dup.copy()

Numero de linhas duplicadas (antes): 143
Numero de linhas duplicadas (depois): 0


**4.3. Tratando valores unicos (espaços e formato)**

Ao analizar os valores unicos das tabelas é possivel ver que há alguns que deveriam representar a mesma coisa, mas que por conta de algum erro nas escrita estão sendo considerados dois valores diferentes. 

In [15]:
silver_df.nunique(dropna=False)

DatGeracaoConjuntoDados             1
NumCNPJAgenteDistribuidora        103
SigAgenteDistribuidora            103
NomAgenteDistribuidora            103
NomTipoMercado                     13
DscModalidadeTarifaria              7
DscSubGrupoTarifario               11
DscClasseConsumoMercado            10
DscSubClasseConsumidor             18
DscDetalheConsumidor               12
IdeAgenteAcessante                  1
NumCNPJAgenteAcessante            103
NomAgenteAcessante               1643
DscPostoTarifario                   5
DscOpcaoEnergia                     5
DscDetalheMercado                  25
DatCompetencia                     28
VlrMercado                    1119081
arquivo_origem                      3
dtype: int64

Não coluna "DscPostoTarifario" por exemplo existe os valores "'Fora ponta' e 'Fora Ponta', que são a mesma coisa, mas um tem o 'P' de ponta maiusculo enquanto o outro não.

Para remover erros como esse basta deixar todas as letras maiusculas ou minusculas.

In [16]:
silver_df["DscPostoTarifario"].dropna().unique()

<StringArray>
['Não se aplica', 'Ponta', 'Fora ponta', 'Intermediário', 'Fora Ponta']
Length: 5, dtype: str

Outro problema que pode ocorrer é terem espaços extras no meio dos textos, exemplo: "Não   se aplica".

Após uma analise é possivel notar que apenas a coluna "NomAgenteAcessante" tem ocorrencia desse problema em especifico.

Para evitar que isso ocorra é necessario substituir esses espaços extras com apenas um espaço.

In [17]:
# Procura por ocorrencia de espaços extras no meio dos textos:
for coluna in silver_df.columns:
    if pd.api.types.is_string_dtype(silver_df[coluna]):
        qtd = (
            silver_df[coluna]
            .str.contains(r"\s{2,}", regex=True, na=False)
            .sum()
        )

        print(f"{coluna}: {qtd}")

NumCNPJAgenteDistribuidora: 0
SigAgenteDistribuidora: 0
NomAgenteDistribuidora: 0
NomTipoMercado: 0
DscModalidadeTarifaria: 0
DscSubGrupoTarifario: 0
DscClasseConsumoMercado: 0
DscSubClasseConsumidor: 0
DscDetalheConsumidor: 0
IdeAgenteAcessante: 0
NumCNPJAgenteAcessante: 0
NomAgenteAcessante: 537
DscPostoTarifario: 0
DscOpcaoEnergia: 0
DscDetalheMercado: 0
arquivo_origem: 0


In [18]:
# Identifica apenas as colunas em que os valores são textos (strings): 
textual_columns = [
    coluna for coluna in silver_df.columns
    if pd.api.types.is_string_dtype(silver_df[coluna])
]

# Normaliza textos para evitar inconsistencias de capitalização/espacos.
for coluna in textual_columns:
    silver_df[coluna] = (
        silver_df[coluna]
        .str.replace(r"\s+", " ", regex=True) # Remove espaços extras no meio do texto.
        .str.strip() # Remove novamente espaços no inicio ou fim, só para garantir que não tem.
        .str.upper() # Evita registros duplicados por diferenças de letras, deixa todas maiusculas.
    )

O codigo acima passa por todas as colunas de texto, removendo os espaços extras e deixando todas as letras maiusculas, resolvendo assim os dois problemas mencionados anteriormente.

Abaixo segue os valores unicos depois do tratamento.

In [19]:
silver_df.nunique(dropna=False)

DatGeracaoConjuntoDados             1
NumCNPJAgenteDistribuidora        103
SigAgenteDistribuidora            103
NomAgenteDistribuidora            103
NomTipoMercado                     13
DscModalidadeTarifaria              7
DscSubGrupoTarifario               11
DscClasseConsumoMercado             9
DscSubClasseConsumidor             18
DscDetalheConsumidor               12
IdeAgenteAcessante                  1
NumCNPJAgenteAcessante            103
NomAgenteAcessante               1642
DscPostoTarifario                   4
DscOpcaoEnergia                     5
DscDetalheMercado                  24
DatCompetencia                     28
VlrMercado                    1119081
arquivo_origem                      3
dtype: int64

As seguinte colunas tiveram o numero de valores unicos reduzidos após o tratamento:

- DscClasseConsumoMercado: 10 -> 9
- NomAgenteAcessante: 1643 -> 1642
- DscPostoTarifario: 5 -> 4
- DscDetalheMercado: 25 -> 24

In [20]:
silver_df["DscPostoTarifario"].dropna().unique()

<StringArray>
['NÃO SE APLICA', 'PONTA', 'FORA PONTA', 'INTERMEDIÁRIO']
Length: 4, dtype: str

In [21]:
# Procura por ocorrencia de espaços extras no meio dos textos:
for coluna in silver_df.columns:
    if pd.api.types.is_string_dtype(silver_df[coluna]):
        qtd = (
            silver_df[coluna]
            .str.contains(r"\s{2,}", regex=True, na=False)
            .sum()
        )

        print(f"{coluna}: {qtd}")

NumCNPJAgenteDistribuidora: 0
SigAgenteDistribuidora: 0
NomAgenteDistribuidora: 0
NomTipoMercado: 0
DscModalidadeTarifaria: 0
DscSubGrupoTarifario: 0
DscClasseConsumoMercado: 0
DscSubClasseConsumidor: 0
DscDetalheConsumidor: 0
IdeAgenteAcessante: 0
NumCNPJAgenteAcessante: 0
NomAgenteAcessante: 0
DscPostoTarifario: 0
DscOpcaoEnergia: 0
DscDetalheMercado: 0
arquivo_origem: 0


**4.4. Tratando valores unicos (acentuação)**

Outro problema em relação a valores unicos que pode ocorrer é o da acentuação, onde dois valores que deveriam ser iguais foram categorizados como diferentes pela falta de acentuação, exemplo: "Não" != "NAO".

Para verificar se há algum problema com a acentuação será criada uma copia do DataFrame onde todos os textos não tem acentuação, após isso será comparado o numero de valores unicos com e sem a acentuação, se o valor se manter igual está tudo ok, mas se o valor mudar é porque existe algum valor em que faltou a acentuação.  

In [22]:
# Função que recebe um texto e retorna o mesmo texto só que sem a acentuação:
def remover_acentos(texto):
    if pd.isna(texto):
        return texto # Se for nulo, retorna.

    return ''.join( # Junta os caracteres.
        c for c in unicodedata.normalize('NFKD', str(texto)) # Separa os caracteres, separando acentuação de letra.
        if not unicodedata.combining(c) # Verifica se é acentuação ou letra, só deixa juntar se for letra.
    )

In [23]:
# --- Essa demora pra rodar --- #

# Cria uma cópia do DataFrame:
df_sem_acento = silver_df.copy()

# Remove acentuação das colunas textuais da copia:
for col in textual_columns:
    df_sem_acento[col] = df_sem_acento[col].apply(remover_acentos)

# Compara quantidade de valores únicos:
for col in textual_columns:
    unicos_original = silver_df[col].nunique(dropna=True)
    unicos_sem_acento = df_sem_acento[col].nunique(dropna=True)

    if unicos_original != unicos_sem_acento:
        print(
            f"{col}: "
            f"{unicos_original} -> {unicos_sem_acento} "
            f"(redução de {unicos_original - unicos_sem_acento})"
        )

NomAgenteAcessante: 1642 -> 1641 (redução de 1)


Na coluna "NomAgenteAcessante" houve uma redução, logo será necessario descobrir qual o nome do acessante que está sem a acentuação e corrigir ele.

In [24]:
# Obtem os valores unicos da coluna "NomAgenteAcessante", com acentuação:
col = "NomAgenteAcessante"
originais = pd.Series(silver_df[col].dropna().unique())

# Cria um DataFrame para comparar os valores originais com os valores sem acentuação.
comparacao = pd.DataFrame({
    "original": originais,
    "sem_acento": originais.apply(remover_acentos)
})

# Agrupa o DataFrame pelos valores sem acentuação e transforma a coluna dos originais em lista;
# O valor que tiver dois valores correspondentes a ele na coluna "original" (um com e outro sem acentuação)
# vai ter uma lista de comprimento 2.
conflitos = (
    comparacao.groupby("sem_acento")["original"].agg(list)
)

# Encontra o conflito flitrando por aquele que tem a lisma maior que 1.
conflitos = conflitos[conflitos.apply(len) > 1]
print(conflitos)

sem_acento
SAO VALENTIM    [SÃO VALENTIM, SAO VALENTIM]
Name: original, dtype: object


Após descobrir qual que está errado basta substituir ele pelo certo.

In [25]:
silver_df["NomAgenteAcessante"] = silver_df["NomAgenteAcessante"].replace("SAO VALENTIM", "SÃO VALENTIM")

In [26]:
# Obtem os valores unicos da coluna "NomAgenteAcessante", com acentuação:
col = "NomAgenteAcessante"
originais = pd.Series(silver_df[col].dropna().unique())

# Cria um DataFrame para comparar os valores originais com os valores sem acentuação.
comparacao = pd.DataFrame({
    "original": originais,
    "sem_acento": originais.apply(remover_acentos)
})

# Agrupa o DataFrame pelos valores sem acentuação e transforma a coluna dos originais em lista;
# O valor que tiver dois valores correspondentes a ele na coluna "original" (um com e outro sem acentuação)
# vai ter uma lista de comprimento 2.
conflitos = (
    comparacao.groupby("sem_acento")["original"].agg(list)
)

# Encontra o conflito flitrando por aquele que tem a lisma maior que 1.
conflitos = conflitos[conflitos.apply(len) > 1]
print(conflitos)

Series([], Name: original, dtype: object)


Conflito vazio significa que o problema foi resolvido.

**4.5. Remove registros**

Caso um registro tenha os valores da coluna "DatCompetencia" ou "VlrMercado" como nulos, o registro deve ser removido, pois esses valores não podem ser substituidos por outro valor qualquer.

O codigo abaixo é o responsavel por remover qualquer registro que tenha um valor nulo em qualquer uma dessas 2 colunas.

In [40]:
# Remove registros sem valor de mercado e sem data de competencia, pois nao sao uteis para agregacao.
silver_df = silver_df.dropna(subset=["DatCompetencia", "VlrMercado"])

**4.6. Colunas auxiliares**

Colunas criadas a partir de colunas já existentes para auxiliar na analise dos dados.

In [51]:
silver_df["DatCompetencia"].unique()

<DatetimeArray>
['2024-01-01 00:00:00', '2024-10-01 00:00:00', '2024-11-01 00:00:00',
 '2024-12-01 00:00:00', '2024-03-01 00:00:00', '2024-02-01 00:00:00',
 '2024-04-01 00:00:00', '2024-05-01 00:00:00', '2024-06-01 00:00:00',
 '2024-07-01 00:00:00', '2024-08-01 00:00:00', '2024-09-01 00:00:00',
 '2025-01-01 00:00:00', '2025-10-01 00:00:00', '2025-11-01 00:00:00',
 '2025-12-01 00:00:00', '2025-03-01 00:00:00', '2025-02-01 00:00:00',
 '2025-04-01 00:00:00', '2025-05-01 00:00:00', '2025-06-01 00:00:00',
 '2025-07-01 00:00:00', '2025-08-01 00:00:00', '2025-09-01 00:00:00',
 '2026-02-01 00:00:00', '2026-01-01 00:00:00', '2026-03-01 00:00:00',
 '2026-04-01 00:00:00']
Length: 28, dtype: datetime64[us]

Como é possivel observar acima, a unica parte importante das datas são os anos e meses, já que o resto não muda, logo serão feitas colunas auxiliares para analise temporal apenas do ano, mes e trimestre apenas.

In [52]:
# Colunas auxiliares para analise temporal.
silver_df["competencia_ano"] = silver_df["DatCompetencia"].dt.year
silver_df["competencia_mes"] = silver_df["DatCompetencia"].dt.to_period("M").astype(str)
silver_df["competencia_trimestre"] = silver_df["DatCompetencia"].dt.to_period("Q").astype(str)

**4.7. Preenchendo nulos**

Preenche os nulos faltantes com valores defaut que indicam que os valores não foram informados.

Preferimos preencher com valor defaut do que remover, já que são muitos registros. 

In [60]:
# Preenche nulos dos atributos de acessante para evitar problemas em filtros de BI.
silver_df["IdeAgenteAcessante"] = silver_df["IdeAgenteAcessante"].fillna("Nao Informado")
silver_df["NomAgenteAcessante"] = silver_df["NomAgenteAcessante"].fillna("Nao Informado")
silver_df["NumCNPJAgenteAcessante"] = silver_df["NumCNPJAgenteAcessante"].fillna("00000000000000")

**4.8. Resultados**

Resultados após todo o tratamento da camada silver.

In [63]:
print(silver_df.isnull().sum())

DatGeracaoConjuntoDados       0
NumCNPJAgenteDistribuidora    0
SigAgenteDistribuidora        0
NomAgenteDistribuidora        0
NomTipoMercado                0
DscModalidadeTarifaria        0
DscSubGrupoTarifario          0
DscClasseConsumoMercado       0
DscSubClasseConsumidor        0
DscDetalheConsumidor          0
IdeAgenteAcessante            0
NumCNPJAgenteAcessante        0
NomAgenteAcessante            0
DscPostoTarifario             0
DscOpcaoEnergia               0
DscDetalheMercado             0
DatCompetencia                0
VlrMercado                    0
arquivo_origem                0
competencia_ano               0
competencia_mes               0
competencia_trimestre         0
dtype: int64


In [64]:
silver_df.head(6)

,DatGeracaoConjuntoDados,NumCNPJAgenteDistribuidora,SigAgenteDistribuidora,NomAgenteDistribuidora,NomTipoMercado,DscModalidadeTarifaria,DscSubGrupoTarifario,DscClasseConsumoMercado,DscSubClasseConsumidor,DscDetalheConsumidor,...,NomAgenteAcessante,DscPostoTarifario,DscOpcaoEnergia,DscDetalheMercado,DatCompetencia,VlrMercado,arquivo_origem,competencia_ano,competencia_mes,competencia_trimestre
0,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,SISTEMA ISOLADO - REGULAR,CONVENCIONAL,B1,RESIDENCIAL,RESIDENCIAL BAIXA RENDA – FAIXA 04,NÃO SE APLICA,...,NÃO SE APLICA,NÃO SE APLICA,CATIVO,ENERGIA TE (KWH),2024-01-01,583151.00,SAMP-2024.CSV,2024,2024-01,2024Q1
1,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,REGULAR,CONVENCIONAL,B1,RESIDENCIAL,RESIDENCIAL BAIXA RENDA – FAIXA 01,NÃO SE APLICA,...,NÃO SE APLICA,NÃO SE APLICA,CATIVO,PIS/PASEP (R$),2024-01-01,5431.29,SAMP-2024.CSV,2024,2024-01,2024Q1
2,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,REGULAR,AZUL,A4,SERVIÇO PÚBLICO,"ÁGUA, ESGOTO E SANEAMENTO",NÃO SE APLICA,...,NÃO SE APLICA,PONTA,CATIVO,RECEITA DEMANDA (R$),2024-01-01,28137.26,SAMP-2024.CSV,2024,2024-01,2024Q1
3,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,SISTEMA DE COMPENSAÇÃO GD II,BRANCA,B1,RESIDENCIAL,RESIDENCIAL,NÃO SE APLICA,...,NÃO SE APLICA,FORA PONTA,CATIVO,RECEITA ENERGIA COMPENSADA (R$),2024-01-01,235.94,SAMP-2024.CSV,2024,2024-01,2024Q1
4,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,REGULAR,CONVENCIONAL,B1,RESIDENCIAL,RESIDENCIAL BAIXA RENDA – FAIXA 04,NÃO SE APLICA,...,NÃO SE APLICA,NÃO SE APLICA,CATIVO,RECEITA ENERGIA (R$),2024-01-01,2061470.44,SAMP-2024.CSV,2024,2024-01,2024Q1
5,2026-05-15,04065033000170,EAC,ENERGISA ACRE - DISTRIBUIDORA DE ENERGIA S.A,REFATURAMENTO - REGULAR,CONVENCIONAL,B3,COMERCIAL,NÃO SE APLICA,NÃO SE APLICA,...,NÃO SE APLICA,NÃO SE APLICA,CATIVO,ICMS (R$),2024-01-01,-418.83,SAMP-2024.CSV,2024,2024-01,2024Q1


In [ ]:
silver_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3029186 entries, 0 to 3029185
Data columns (total 22 columns):
 #   Column                      Dtype         
---  ------                      -----         
 0   DatGeracaoConjuntoDados     datetime64[us]
 1   NumCNPJAgenteDistribuidora  str           
 2   SigAgenteDistribuidora      str           
 3   NomAgenteDistribuidora      str           
 4   NomTipoMercado              str           
 5   DscModalidadeTarifaria      str           
 6   DscSubGrupoTarifario        str           
 7   DscClasseConsumoMercado     str           
 8   DscSubClasseConsumidor      str           
 9   DscDetalheConsumidor        str           
 10  IdeAgenteAcessante          str           
 11  NumCNPJAgenteAcessante      str           
 12  NomAgenteAcessante          str           
 13  DscPostoTarifario           str           
 14  DscOpcaoEnergia             str           
 15  DscDetalheMercado           str           
 16  DatCompetencia              d

In [62]:
# Indicadores de qualidade da camada Silver.
resumo_qualidade = pd.DataFrame({
    "metric": [
        "linhas_bronze",
        "linhas_silver",
        "duplicatas_removidas",
        "linhas_sem_competencia_ou_valor",
        "linhas_descartadas_por_nulos",
        "datas_competencia_invalidas",
        "valores_nulos_vlrmercado",
    ],
    "value": [
        len(df),
        len(silver_df),
        linhas_duplicadas_exatas,
        linhas_sem_competencia_ou_valor,
        linhas_sem_competencia_ou_valor,
        int(df["DatCompetencia"].isna().sum()),
        int(df["VlrMercado"].isna().sum()),
    ],
})

resumo_qualidade

NameError: name 'linhas_sem_competencia_ou_valor' is not defined

**4. Camada Gold - esquema estrela**

Nesta etapa transformamos a camada Silver em dimensões e fato para consultas analíticas mais consistentes.

In [15]:
processed_root = arquivos_path / "processed"
bronze_dir = processed_root / "bronze"
silver_dir = processed_root / "silver"
gold_dir = processed_root / "gold"

bronze_dir.mkdir(parents=True, exist_ok=True)
silver_dir.mkdir(parents=True, exist_ok=True)
gold_dir.mkdir(parents=True, exist_ok=True)

bronze_path = bronze_dir / "bronze_dataset.csv"
silver_path = silver_dir / "silver_dataset.csv"
dim_tempo_path = gold_dir / "dim_tempo.csv"
dim_distribuidora_path = gold_dir / "dim_distribuidora.csv"
dim_acessante_path = gold_dir / "dim_acessante.csv"
dim_mercado_path = gold_dir / "dim_mercado.csv"
fato_energia_path = gold_dir / "fato_energia.csv"

# Dimensão tempo
dim_tempo = (
    silver_df[["DatCompetencia"]]
    .drop_duplicates()
    .dropna()
    .sort_values("DatCompetencia")
    .reset_index(drop=True)
    .assign(
        tempo_id=lambda tabela: tabela.index + 1,
        ano=lambda tabela: tabela["DatCompetencia"].dt.year,
        mes=lambda tabela: tabela["DatCompetencia"].dt.month,
        nome_mes=lambda tabela: tabela["DatCompetencia"].dt.strftime("%B"),
        trimestre=lambda tabela: tabela["DatCompetencia"].dt.quarter,
        semestre=lambda tabela: ((tabela["DatCompetencia"].dt.month - 1) // 6) + 1,
    )
    [["tempo_id", "DatCompetencia", "ano", "mes", "nome_mes", "trimestre", "semestre"]]
    .rename(columns={"DatCompetencia": "data_competencia"})
)

# Dimensão distribuidora

dim_distribuidora = (
    silver_df[["NumCNPJAgenteDistribuidora", "SigAgenteDistribuidora", "NomAgenteDistribuidora"]]
    .drop_duplicates()
    .sort_values(["SigAgenteDistribuidora", "NomAgenteDistribuidora"])
    .reset_index(drop=True)
    .assign(distribuidora_id=lambda tabela: tabela.index + 1)
    [["distribuidora_id", "NumCNPJAgenteDistribuidora", "SigAgenteDistribuidora", "NomAgenteDistribuidora"]]
)

# Dimensão acessante

dim_acessante = (
    silver_df[["NumCNPJAgenteAcessante", "IdeAgenteAcessante", "NomAgenteAcessante"]]
    .drop_duplicates()
    .sort_values(["NomAgenteAcessante", "NumCNPJAgenteAcessante"])
    .reset_index(drop=True)
    .assign(acessante_id=lambda tabela: tabela.index + 1)
    [["acessante_id", "NumCNPJAgenteAcessante", "IdeAgenteAcessante", "NomAgenteAcessante"]]
)

# Dimensão mercado

dim_mercado = (
    silver_df[[
        "NomTipoMercado",
        "DscModalidadeTarifaria",
        "DscSubGrupoTarifario",
        "DscClasseConsumoMercado",
        "DscSubClasseConsumidor",
        "DscDetalheConsumidor",
        "DscPostoTarifario",
        "DscOpcaoEnergia",
        "DscDetalheMercado",
    ]]
    .drop_duplicates()
    .sort_values([
        "NomTipoMercado",
        "DscModalidadeTarifaria",
        "DscPostoTarifario",
        "DscDetalheMercado",
    ])
    .reset_index(drop=True)
    .assign(mercado_id=lambda tabela: tabela.index + 1)
    [[
        "mercado_id",
        "NomTipoMercado",
        "DscModalidadeTarifaria",
        "DscSubGrupoTarifario",
        "DscClasseConsumoMercado",
        "DscSubClasseConsumidor",
        "DscDetalheConsumidor",
        "DscPostoTarifario",
        "DscOpcaoEnergia",
        "DscDetalheMercado",
    ]]
)

# Tabela fato
fato_energia = silver_df[[
    "DatGeracaoConjuntoDados",
    "DatCompetencia",
    "NumCNPJAgenteDistribuidora",
    "SigAgenteDistribuidora",
    "NomAgenteDistribuidora",
    "NumCNPJAgenteAcessante",
    "IdeAgenteAcessante",
    "NomAgenteAcessante",
    "NomTipoMercado",
    "DscModalidadeTarifaria",
    "DscSubGrupoTarifario",
    "DscClasseConsumoMercado",
    "DscSubClasseConsumidor",
    "DscDetalheConsumidor",
    "DscPostoTarifario",
    "DscOpcaoEnergia",
    "DscDetalheMercado",
    "VlrMercado",
    "arquivo_origem",
]].copy()

fato_energia = fato_energia.merge(
    dim_tempo[["tempo_id", "data_competencia"]],
    left_on="DatCompetencia",
    right_on="data_competencia",
    how="left",
    validate="m:1",
)
fato_energia = fato_energia.merge(
    dim_distribuidora[["distribuidora_id", "NumCNPJAgenteDistribuidora", "SigAgenteDistribuidora", "NomAgenteDistribuidora"]],
    on=["NumCNPJAgenteDistribuidora", "SigAgenteDistribuidora", "NomAgenteDistribuidora"],
    how="left",
    validate="m:1",
)
fato_energia = fato_energia.merge(
    dim_acessante[["acessante_id", "NumCNPJAgenteAcessante", "IdeAgenteAcessante", "NomAgenteAcessante"]],
    on=["NumCNPJAgenteAcessante", "IdeAgenteAcessante", "NomAgenteAcessante"],
    how="left",
    validate="m:1",
)
fato_energia = fato_energia.merge(
    dim_mercado[[
        "mercado_id",
        "NomTipoMercado",
        "DscModalidadeTarifaria",
        "DscSubGrupoTarifario",
        "DscClasseConsumoMercado",
        "DscSubClasseConsumidor",
        "DscDetalheConsumidor",
        "DscPostoTarifario",
        "DscOpcaoEnergia",
        "DscDetalheMercado",
    ]],
    on=[
        "NomTipoMercado",
        "DscModalidadeTarifaria",
        "DscSubGrupoTarifario",
        "DscClasseConsumoMercado",
        "DscSubClasseConsumidor",
        "DscDetalheConsumidor",
        "DscPostoTarifario",
        "DscOpcaoEnergia",
        "DscDetalheMercado",
    ],
    how="left",
    validate="m:1",
)

fato_energia = fato_energia.drop(columns=["data_competencia"])
fato_energia = fato_energia.assign(fato_id=lambda tabela: tabela.index + 1)[[
    "fato_id",
    "tempo_id",
    "distribuidora_id",
    "acessante_id",
    "mercado_id",
    "DatGeracaoConjuntoDados",
    "VlrMercado",
    "arquivo_origem",
]]

# Audotoria de integridade das chaves primarias

print("Auditoria de integridade (verificar chaves unicas)")

dimensoes_checagem = {
    "Dim Distribuidora": (dim_distribuidora, "distribuidora_id"),
    "Dim Acessante": (dim_acessante, "acessante_id"),
    "Dim Mercado": (dim_mercado, "mercado_id"),
    "Dim Tempo": (dim_tempo, "tempo_id")
}

for nome_dim, (dataframe, coluna_chave) in dimensoes_checagem.items():
    total_linhas = len(dataframe)
    valores_unicos = dataframe[coluna_chave].nunique()
    possui_nulos = dataframe[coluna_chave].isnull().any()
    
    if total_linhas == valores_unicos and not possui_nulos:
        print(f"[OK] {nome_dim}: Chave '{coluna_chave}' é integra (Unica e sem nulos).")
    else:
        print(f"[ALERTA CRITICO] {nome_dim}: Falha de integridade! Linhas: {total_linhas} | Unicos: {valores_unicos} | Contem nulos: {possui_nulos}")

# Validacao de relacionamento fato x dimensao
print("\nVerificacao de consistencia da tabela fato")
chaves_fato = ["tempo_id", "distribuidora_id", "acessante_id", "mercado_id"]

for chave in chaves_fato:
    nulos_na_fato = fato_energia[chave].isnull().sum()
    if nulos_na_fato > 0:
        print(f"[ALERTA] A coluna '{chave}' possui {nulos_na_fato} registros sem correspondencia nas dimensoes!")
    else:
        print(f"[OK] Integração da chave '{chave}' realizada com sucesso.")
        

# Exporta Bronze, Silver e o esquema estrela com CSV separado por virgula.
df.to_csv(bronze_path, index=False, encoding="utf-8-sig")
silver_df.to_csv(silver_path, index=False, encoding="utf-8-sig")
dim_tempo.to_csv(dim_tempo_path, index=False, encoding="utf-8-sig")
dim_distribuidora.to_csv(dim_distribuidora_path, index=False, encoding="utf-8-sig")
dim_acessante.to_csv(dim_acessante_path, index=False, encoding="utf-8-sig")
dim_mercado.to_csv(dim_mercado_path, index=False, encoding="utf-8-sig")
fato_energia.to_csv(fato_energia_path, index=False, encoding="utf-8-sig")

{
    "bronze_path": str(bronze_path),
    "silver_path": str(silver_path),
    "dim_tempo_path": str(dim_tempo_path),
    "dim_distribuidora_path": str(dim_distribuidora_path),
    "dim_acessante_path": str(dim_acessante_path),
    "dim_mercado_path": str(dim_mercado_path),
    "fato_energia_path": str(fato_energia_path),
}

Auditoria de integridade (verificar chaves unicas)
[OK] Dim Distribuidora: Chave 'distribuidora_id' é integra (Unica e sem nulos).
[OK] Dim Acessante: Chave 'acessante_id' é integra (Unica e sem nulos).
[OK] Dim Mercado: Chave 'mercado_id' é integra (Unica e sem nulos).
[OK] Dim Tempo: Chave 'tempo_id' é integra (Unica e sem nulos).

Verificacao de consistencia da tabela fato
[OK] Integração da chave 'tempo_id' realizada com sucesso.
[OK] Integração da chave 'distribuidora_id' realizada com sucesso.
[OK] Integração da chave 'acessante_id' realizada com sucesso.
[OK] Integração da chave 'mercado_id' realizada com sucesso.


{'bronze_path': '../Datas/processed/bronze/bronze_dataset.csv',
 'silver_path': '../Datas/processed/silver/silver_dataset.csv',
 'dim_tempo_path': '../Datas/processed/gold/dim_tempo.csv',
 'dim_distribuidora_path': '../Datas/processed/gold/dim_distribuidora.csv',
 'dim_acessante_path': '../Datas/processed/gold/dim_acessante.csv',
 'dim_mercado_path': '../Datas/processed/gold/dim_mercado.csv',
 'fato_energia_path': '../Datas/processed/gold/fato_energia.csv'}

**5. Validação final do ETL**

Aqui conferimos o volume final das dimensões e da tabela fato geradas no esquema estrela.

In [16]:
print(f"Bronze linhas: {len(df):,}")
print(f"Silver linhas: {len(silver_df):,}")
print(f"Dim tempo linhas: {len(dim_tempo):,}")
print(f"Dim distribuidora linhas: {len(dim_distribuidora):,}")
print(f"Dim acessante linhas: {len(dim_acessante):,}")
print(f"Dim mercado linhas: {len(dim_mercado):,}")
print(f"Fato energia linhas: {len(fato_energia):,}")

display(dim_tempo.head())
display(dim_distribuidora.head())
display(dim_acessante.head())
display(dim_mercado.head())
display(fato_energia.head())

Bronze linhas: 3,029,329
Silver linhas: 3,029,186
Dim tempo linhas: 28
Dim distribuidora linhas: 103
Dim acessante linhas: 1,643
Dim mercado linhas: 12,902
Fato energia linhas: 3,029,186


,tempo_id,data_competencia,ano,mes,nome_mes,trimestre,semestre
0,1,2024-01-01,2024,1,January,1,1
1,2,2024-02-01,2024,2,February,1,1
2,3,2024-03-01,2024,3,March,1,1
3,4,2024-04-01,2024,4,April,2,1
4,5,2024-05-01,2024,5,May,2,1


,distribuidora_id,NumCNPJAgenteDistribuidora,SigAgenteDistribuidora,NomAgenteDistribuidora
0,1,02341467000120,AME,AMAZONAS ENERGIA S.A
1,2,02341470000144,BOA VISTA,RORAIMA ENERGIA S.A.
2,3,30460297000139,CASTRO-DIS,COOPERATIVA DE DISTRIBUICAO DE ENERGIA ELETRIC...
3,4,05965546000109,CEA,COMPANHIA DE ELETRICIDADE DO AMAPA CEA
4,5,60196987000193,CEDRAP,COOPERATIVA DE ELETRIFICACAO DA REGIAO DO ALTO...


,acessante_id,NumCNPJAgenteAcessante,IdeAgenteAcessante,NomAgenteAcessante
0,1,0,Nao Informado,ABAÚNA
1,2,0,Nao Informado,ABRANJO I
2,3,0,Nao Informado,ACOMAIS
3,4,0,Nao Informado,ADO POPINHAKI
4,5,0,Nao Informado,ADVOGADO EDUARDO SOARES (ANTIGA BOA ESPERANÇA)


,mercado_id,NomTipoMercado,DscModalidadeTarifaria,DscSubGrupoTarifario,DscClasseConsumoMercado,DscSubClasseConsumidor,DscDetalheConsumidor,DscPostoTarifario,DscOpcaoEnergia,DscDetalheMercado
0,1,REFATURAMENTO - REGULAR,AZUL,A4,SERVIÇO PÚBLICO,"ÁGUA, ESGOTO E SANEAMENTO",NÃO SE APLICA,FORA PONTA,CATIVO,DEMANDA FATURADA (KW)
1,2,REFATURAMENTO - REGULAR,AZUL,A3,INDUSTRIAL,NÃO SE APLICA,NÃO SE APLICA,FORA PONTA,LIVRE,DEMANDA FATURADA (KW)
2,3,REFATURAMENTO - REGULAR,AZUL,A3A,COMERCIAL,NÃO SE APLICA,NÃO SE APLICA,FORA PONTA,CATIVO,DEMANDA FATURADA (KW)
3,4,REFATURAMENTO - REGULAR,AZUL,A2,SERVIÇO PÚBLICO,TRAÇÃO ELÉTRICA,NÃO SE APLICA,FORA PONTA,CATIVO,DEMANDA FATURADA (KW)
4,5,REFATURAMENTO - REGULAR,AZUL,A4,PODER PÚBLICO,NÃO SE APLICA,NÃO SE APLICA,FORA PONTA,CATIVO,DEMANDA FATURADA (KW)


,fato_id,tempo_id,distribuidora_id,acessante_id,mercado_id,DatGeracaoConjuntoDados,VlrMercado,arquivo_origem
0,1,1,70,1009,12572,2026-05-15,583151.00,SAMP-2024.CSV
1,2,1,70,1009,3633,2026-05-15,5431.29,SAMP-2024.CSV
2,3,1,70,1009,3036,2026-05-15,28137.26,SAMP-2024.CSV
3,4,1,70,1009,8557,2026-05-15,235.94,SAMP-2024.CSV
4,5,1,70,1009,3686,2026-05-15,2061470.44,SAMP-2024.CSV


In [ ]:
#Instalando as dependências necessárias
!pip install sqlalchemy
!pip install psycopg2-binary

In [ ]:
from sqlalchemy import create_engine

# Preencher com Dados no Postgree

usuario_db = 'seu_usuario'           
senha_db = 'sua_senha'               
host_db = 'seu_host'                 
porta_db = 'sua_porta'               
nome_banco_db = 'seu_nome_do_banco'

# Criando a string de conexao
DATABASE_URL = f'postgresql://{usuario_db}:{senha_db}@{host_db}:{porta_db}/{nome_banco_db}?client_encoding=utf8'

print("Tentando criar conexão com o banco PostgreSQL...")

try:
    # Estabelecendo conexão ativa
    engine = create_engine(DATABASE_URL)
    print("Conexao bem-sucedida! Iniciando a carga do Esquema Estrela no Postgres...\n")
    
    # Carregando as  Dimensoes no Postgres
    
    print("Carregando Dimensao Tempo...")
    dim_tempo.to_sql('dim_tempo', con=engine, if_exists='replace', index=False)
    
    print("Carregando Dimensao Distribuidora...")
    dim_distribuidora.to_sql('dim_distribuidora', con=engine, if_exists='replace', index=False)
    
    print("Carregando Dimensao Acessante...")
    dim_acessante.to_sql('dim_acessante', con=engine, if_exists='replace', index=False)
    
    print("Carregando Dimensao Mercado...")
    dim_mercado.to_sql('dim_mercado', con=engine, if_exists='replace', index=False)
    
    # Carregando Tabela Fato no Postgres
    
    print("Carregando Tabela Fato Energia...")
    fato_energia.to_sql('fato_energia', con=engine, if_exists='replace', index=False)

    print("\n CARGA CONCLUÍDA! Todas as tabelas do Esquema Estrela foram criadas no seu PostgreSQL.")

except Exception as e:
    print(f"\n--- [ERRO CRÍTICO] ---")
    print(f"Erro ao conectar ou carregar os dados no PostgreSQL: {e}")